# Data inspection

Read-only inspection of the Stage 00 CSV handoff. Use the pinned PyArrow 25.0.1 and scikit-learn 1.9.1 environment documented in README.md. Run after download and preparation.

Checks cover source integrity, labels, duplicates, approvals, and preserved dialogue content. Model-oriented analysis belongs to later plans. Candidate reviews are attributed in preparation_decisions.json; assistant review is not independent human verification.

In [ ]:
from pathlib import Path
from collections import Counter
import csv
import sys

ROOT = Path.cwd()
if ROOT.name == "02 - DATASETs":
    ROOT = ROOT.parent
DATA = ROOT / "02 - DATASETs"
sys.path.insert(0, str(DATA))
import prepare

manifest, rows = prepare.load_sources()
approvals = prepare.review_inputs(manifest, rows)
print("Raw counts:", dict(Counter(r["source_dataset"] for r in rows)))
print("Project label counts before exclusions:", dict(Counter(r["label"] for r in rows)))
print("Categories:", dict(Counter((r["source_dataset"], r["original_category"]) for r in rows)))
print("Blank transcripts:", sum(not r["transcript"].strip() for r in rows))
print("Replacement-character rows:", [r["sample_id"] for r in rows if "\ufffd" in r["transcript"]])
print("Integrity dispositions:", {cid: d["decision"] for cid, d in approvals["integrity"].items()})


In [ ]:
pairs = prepare.candidates(rows)
reviews = prepare.resolve_pairs(pairs, approvals)
kept, audit = prepare.prepare_records(rows, approvals, pairs)
print("Candidate pairs:", len(pairs))
print("Relationship outcomes:", dict(Counter(r["decision"] for r in reviews.values())))
print("Final dispositions:", dict(Counter(r["action"] for r in audit if r["action"] in ("retain", "exclude", "remove_duplicate"))))
print("Retained by source:", dict(Counter(r["source_dataset"] for r in kept)))
print("Retained labels:", dict(Counter(r["label"] for r in kept)))
assert all(r["transcript"] == next(raw["transcript"] for raw in rows if raw["sample_id"] == r["sample_id"]) for r in kept)


In [ ]:
with (DATA / "prepared/prepared.csv").open(encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    assert reader.fieldnames == prepare.COLUMNS
    saved = list(reader)
expected = [{key: str(r[key]) for key in prepare.COLUMNS} for r in kept]
assert saved == expected
print("Canonical CSV matches reviewed preparation:", len(saved), "records")
print("No raw changes, split assignments, or model preprocessing were made.")
